<a href="https://colab.research.google.com/github/santhiya2313-spec/MachineLearning-Lab/blob/main/ML_Day14New3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()

Saving student_ml_dataset_100_records.csv to student_ml_dataset_100_records (1).csv


In [2]:
import pandas as pd

df = pd.read_csv("student_ml_dataset_100_records.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (100, 8)


,StudentID,Name,Study_Hours,Attendance_,Previous_Score,Internet_Access,Extracurricular,FinalGrade
0,101,Student_1,6,97,67,Yes,No,64
1,102,Student_2,8,65,77,Yes,No,72
2,103,Student_3,-3,82,91,Yes,Yes,83
3,104,Student_4,8,96,71,Yes,Yes,66
4,105,Student_5,6,75,70,No,No,60


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pickle

# Remove unnecessary columns
data = df.drop(columns=["StudentID", "Name"])

# Fix invalid values
data.loc[data["Study_Hours"] < 0, "Study_Hours"] = data["Study_Hours"].median()
data.loc[data["Attendance_"] > 100, "Attendance_"] = data["Attendance_"].median()

# Encode categorical columns
le1 = LabelEncoder()
le2 = LabelEncoder()

data["Internet_Access"] = le1.fit_transform(data["Internet_Access"])
data["Extracurricular"] = le2.fit_transform(data["Extracurricular"])

# Create grade classes
data["Grade_Class"] = pd.cut(
    data["FinalGrade"],
    bins=[0, 70, 85, 100],
    labels=["Class C", "Class B", "Class A"]
)

# Features
X = data[
    [
        "Study_Hours",
        "Attendance_",
        "Previous_Score",
        "Internet_Access",
        "Extracurricular"
    ]
]

# Target
y = data["Grade_Class"]

# Train model
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X, y)

print("Model trained successfully!")

Model trained successfully!


In [4]:
with open("model.pkl", "wb") as file:
    pickle.dump(model, file)

print("model.pkl saved successfully!")

model.pkl saved successfully!


In [5]:
app_code = '''
from flask import Flask, request, jsonify
import numpy as np
import pickle

app = Flask(__name__)

# Load trained model
with open("model.pkl", "rb") as file:
    model = pickle.load(file)

@app.route("/predict", methods=["POST"])
def predict():

    data = request.get_json()

    features = np.array(data["features"]).reshape(1, -1)

    prediction = model.predict(features)

    return jsonify({
        "prediction": prediction[0]
    })

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''

with open("app.py", "w") as file:
    file.write(app_code)

print("app.py created successfully!")

app.py created successfully!


In [6]:
requirements = """
Flask
numpy
pandas
scikit-learn
"""

with open("requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created!")

requirements.txt created!


In [7]:
dockerfile = """
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]
"""

with open("Dockerfile", "w") as file:
    file.write(dockerfile)

print("Dockerfile created successfully!")

Dockerfile created successfully!


In [8]:
import os

print(os.listdir())

['.config', '.ipynb_checkpoints', 'requirements.txt', 'student_ml_dataset_100_records.csv', 'Dockerfile', 'app.py', 'model.pkl', 'sample_data']


In [9]:
!pip install flask

In [10]:
import subprocess

process = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("Flask server started!")

Flask server started!


In [11]:
import requests

url = "http://127.0.0.1:5000/predict"

data = {
    "features": [8, 90, 85, 1, 1]
}

response = requests.post(url, json=data)

print(response.json())

{'prediction': 'Class B'}


In [14]:
print(open("Dockerfile").read())


FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]



In [15]:
print("Docker Build Command:")
print("docker build -t ml-flask-app .")

Docker Build Command:
docker build -t ml-flask-app .


In [16]:
print("Docker Run Command:")
print("docker run -p 5000:5000 ml-flask-app")

Docker Run Command:
docker run -p 5000:5000 ml-flask-app


In [17]:
import requests

url = "http://127.0.0.1:5000/predict"

data = {
    "features": [8, 90, 85, 1, 1]
}

response = requests.post(url, json=data)

print("API Response:")
print(response.json())

API Response:
{'prediction': 'Class B'}
